# IKG Table Lineage Auto Refresh

Use this notebook to regenerate the lineage Excel report from the GitLab SQL scripts and push the output into `sandbox_prj_smart_insights.ikg_table_lineage_metadata_auto_refresh`. The code mirrors the standalone Python script and is safe to rerun whenever the SQL sources change.

In [ ]:
import datetime
import logging

from ikg_table_lineage_auto_refresh import (
    EXCLUDE_FOLDER,
    GitLabSQLFetcher,
    SQLParser,
    LineageBuilder,
    DatabaseUploader,
    rows_to_dataframe,
    write_to_excel,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
)

print("Notebook logger initialized.")

In [ ]:
import getpass

PRIVATE_TOKEN = getpass.getpass("Enter your private token: ")
DB_PASSWORD = getpass.getpass("Enter Password for DB User: ")

In [ ]:
exclude_folders = [folder for folder in EXCLUDE_FOLDER.split() if folder]
fetcher = GitLabSQLFetcher(private_token=PRIVATE_TOKEN, exclude_folders=exclude_folders)
parser = SQLParser()
builder = LineageBuilder(fetcher, parser)

run_ts = datetime.datetime.utcnow()
output_file = write_to_excel(rows_to_dataframe([]), run_timestamp=run_ts)

print(f"Initialized Excel snapshot at {output_file}")


def flush_excel(current_rows):
    snapshot_df = rows_to_dataframe(current_rows)
    write_to_excel(snapshot_df, output_path=output_file)


rows = builder.build(progress_callback=flush_excel, run_timestamp=run_ts)
df = rows_to_dataframe(rows)
print(f"Collected {len(df)} lineage rows from GitLab.")
df.head()

In [ ]:
print("Excel report is being updated at:")
output_file

In [ ]:
db_config = {
    "host": "greenplum-rdsp.zur.swissbank.com",
    "port": "5432",
    "dbname": "gprdsp",
    "user": "ds_rdsp_dev",
    "password": DB_PASSWORD,
}

db_uploader = DatabaseUploader(db_config)
db_uploader.refresh_table(rows)
print("Database table refreshed successfully.")